# Parametrization Or Dynamic Notebook

In [0]:
dbutils.widgets.text("file_name","")

In [0]:
p_file_name = dbutils.widgets.get("file_name")

# Data Reading

In [0]:
df_orders = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", f"abfss://bronze-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/checkPoint_{p_file_name.capitalize()}")\
                .load(f"abfss://raw-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/{p_file_name}")
                # .option("cloudFiles.schemaHints", "order_id string, product_id string") # schemaHints = manual corrections when autopilot gets confused

# Writing Data to Bronze Layer


# 📌 Spark Structured Streaming — `writeStream` Deep Dive

This guide explains the most commonly used methods in `writeStream` with clear concepts and examples.

---

## 🔹 1. What is `writeStream`?

`writeStream` is used to **write streaming DataFrames** (continuous/incremental data) to a destination.

```python
df.writeStream
```

👉 Unlike `write`, this runs continuously and processes new data as it arrives.

---

## 🔹 2. `.format()`

Defines the **sink type (destination)**.

### Common formats:

* `"delta"` → Delta Lake (most common in Databricks)
* `"parquet"`
* `"csv"`
* `"json"`
* `"console"` → debugging
* `"memory"` → temporary table

```python
df.writeStream.format("delta")
```

---

## 🔹 3. `.outputMode()`

Controls **how data is written to sink**.

### Modes:

### 🟢 Append (most used)

* Writes **only new rows**
* Works when no aggregation OR append-safe aggregation

```python
.outputMode("append")
```

---

### 🟡 Update

* Writes **only updated rows**
* Used in aggregations

```python
.outputMode("update")
```

---

### 🔴 Complete

* Writes **entire result table every time**
* Expensive

```python
.outputMode("complete")
```

---

## 🔹 4. `.option()`

Used to configure behavior.

### Important options:

### 📍 `checkpointLocation` (VERY IMPORTANT)

* Stores progress & state
* Required for fault tolerance

```python
.option("checkpointLocation", "/mnt/checkpoints/orders")
```

---

### 📍 `path` (for file sinks)

```python
.option("path", "/mnt/delta/orders")
```

---

### 📍 Other useful options:

* `"mergeSchema"` → handle schema evolution
* `"overwriteSchema"` → overwrite schema

---

## 🔹 5. `.trigger()`

Controls **when data is processed**.

### Types:

### 🔁 Default (micro-batch)

* Runs continuously

---

### ⏱️ Processing time trigger

```python
.trigger(processingTime="10 seconds")
```

👉 Runs every 10 seconds

---

### ⚡ Once trigger

```python
.trigger(once=True)
```

👉 Runs only once (batch-like)

---

### 🚀 Available now (Databricks optimized)

```python
.trigger(availableNow=True)
```

👉 Processes all available data then stops

---

## 🔹 6. `.partitionBy()`

Used to **partition output data** (for performance).

```python
.partitionBy("order_date")
```

👉 Improves query performance

---

## 🔹 7. `.queryName()`

Assigns a name to the streaming query.

```python
.queryName("orders_stream")
```

👉 Useful for:

* Monitoring
* Debugging
* Query management

---

## 🔹 8. `.start()`

Starts the streaming query.

```python
.start("/mnt/delta/orders")
```

OR

```python
.start()
```

(if path is defined in `.option()`)

---

## 🔹 9. `.awaitTermination()`

Keeps the stream running.

```python
query.awaitTermination()
```

👉 Without this, job may exit early

---

## 🔹 10. `.foreachBatch()` (VERY IMPORTANT)

Used for **custom logic per batch**.

```python
def process_batch(df, batch_id):
    df.write.format("delta").mode("append").save("/mnt/delta/orders")

df.writeStream.foreachBatch(process_batch).start()
```

👉 Use cases:

* Upserts (MERGE)
* Writing to multiple sinks
* Complex transformations

---

## 🔹 11. `.foreach()` (Row-level processing)

Less common, used for custom row logic.

```python
df.writeStream.foreach(custom_writer)
```

---

## 🔹 12. `.option("mergeSchema", "true")`

Used when schema evolves.

```python
.option("mergeSchema", "true")
```

---

## 🔹 13. Example End-to-End

```python
df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/mnt/checkpoints/orders") \
    .option("path", "/mnt/delta/orders") \
    .trigger(processingTime="30 seconds") \
    .partitionBy("order_date") \
    .queryName("orders_stream") \
    .start()
```

---

## 🔹 14. Key Best Practices

✔ Always set `checkpointLocation`
✔ Prefer `"append"` mode
✔ Use `"delta"` format in Databricks
✔ Use `foreachBatch` for complex logic
✔ Partition wisely (avoid too many small files)

---

## 🔹 15. Common Interview Questions

### ❓ Difference between `append` and `update`

* Append → only new rows
* Update → changed rows

### ❓ Why checkpoint is required?

* Fault tolerance
* Exactly-once processing

### ❓ When to use `foreachBatch`?

* Upserts
* External system writes

---

## 🎯 Summary

`writeStream` = core API to send streaming data to storage.

Key pillars:

* Format
* Output mode
* Checkpointing
* Trigger
* Sink

---

💡 Tip: In Databricks, **Auto Loader + writeStream (Delta)** is the most common production pattern.



# 🔥 `outputMode` in Spark Structured Streaming — Deep Dive

`outputMode` defines **what data gets written to the sink in every micro-batch**.

---

## 🧠 Core Idea

Every streaming query maintains a **result table (logical state)**.

👉 `outputMode` decides:

> Which part of that table should be written after each trigger?

---

# 🔹 1. APPEND MODE (Most Used)

## ✅ Concept

Only **new rows** are written.

👉 Already written rows are **never touched again**.

---

## 📊 Example 1: Simple Streaming (No Aggregation)

```python
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .load("/input")

df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .start("/output")
```

### 🧾 What happens?

| Batch | Incoming Data | Output |
| ----- | ------------- | ------ |
| 1     | A, B          | A, B   |
| 2     | C             | C      |
| 3     | D             | D      |

👉 Only new records are added

---

## ⚠️ Limitation

❌ Doesn't work with **aggregations without watermark**

---

## 📊 Example 2: Aggregation WITH Watermark

```python
from pyspark.sql.functions import window

df = input_df \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(window("event_time", "5 minutes")) \
    .count()

df.writeStream.outputMode("append")
```

👉 Rows are written **only when aggregation is final (window closed)**

---

# 🔹 2. UPDATE MODE

## ✅ Concept

Only **rows that changed in current batch** are written.

👉 Previously written rows **can be updated**

---

## 📊 Example: Running Count

```python
df = input_df.groupBy("category").count()

df.writeStream \
    .format("console") \
    .outputMode("update") \
    .start()
```

---

### 🧾 What happens?

| Batch | Data | Result Table | Output   |
| ----- | ---- | ------------ | -------- |
| 1     | A,A  | A=2          | A=2      |
| 2     | A,B  | A=3, B=1     | A=3, B=1 |
| 3     | B    | A=3, B=2     | B=2      |

👉 Only **changed rows** are emitted

---

## ⚠️ Key Notes

* Efficient for aggregations
* Not supported by all sinks (e.g., some file formats)

---

# 🔹 3. COMPLETE MODE

## ✅ Concept

Writes **entire result table every time**

---

## 📊 Example: Aggregation

```python
df = input_df.groupBy("category").count()

df.writeStream \
    .format("console") \
    .outputMode("complete") \
    .start()
```

---

### 🧾 What happens?

| Batch | Result Table | Output   |
| ----- | ------------ | -------- |
| 1     | A=2          | A=2      |
| 2     | A=3, B=1     | A=3, B=1 |
| 3     | A=3, B=2     | A=3, B=2 |

👉 Entire table rewritten every batch

---

## ❌ Downsides

* Very expensive
* Not scalable for large data

---

# 🔥 Side-by-Side Comparison

| Feature             | Append        | Update       | Complete           |
| ------------------- | ------------- | ------------ | ------------------ |
| Writes new rows     | ✅ Yes         | ❌ No         | ❌ No               |
| Writes updated rows | ❌ No          | ✅ Yes        | ✅ Yes              |
| Writes full table   | ❌ No          | ❌ No         | ✅ Yes              |
| Best for            | Raw ingestion | Aggregations | Small aggregations |
| Performance         | 🚀 Fast       | ⚡ Medium     | 🐢 Slow            |

---

# 🧠 When to Use What?

## ✅ Use APPEND when:

* Ingesting raw data (Auto Loader)
* No aggregation OR watermark used

👉 Most production pipelines use this

---

## ✅ Use UPDATE when:

* Aggregations (counts, sums)
* Want incremental updates

---

## ✅ Use COMPLETE when:

* Small datasets
* Need full snapshot every time

---

# ⚠️ Important Rule (Interview Favorite)

👉 Without watermark:

* Append ❌ not allowed for aggregation
* Update ✅ allowed
* Complete ✅ allowed

👉 With watermark:

* Append ✅ allowed

---

# 🔥 Real Production Example

## Bronze Layer (Raw Ingestion)

```python
df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .start("/bronze/orders")
```

---

## Silver Layer (Aggregation)

```python
agg_df.writeStream \
    .format("delta") \
    .outputMode("update") \
    .start("/silver/orders_agg")
```

---

# 🎯 Final Intuition

* **Append → “Just add new data”**
* **Update → “Update what changed”**
* **Complete → “Rewrite everything”**

---

💡 If you're working with Databricks in real projects:
👉 80% cases = `append`
👉 15% cases = `update`
👉 5% cases = `complete`

---

If you want next:

* 💥 Watermark + outputMode deep combo (very important)
* 💥 Why append fails without watermark (internals)
* 💥 Interview tricky scenarios

Just tell me 👍


## Adding Metadata Columns

**_metadata is a struct column automatically added by Auto Loader / readStream.**

It contains fields like:

- file_path
- file_name
- file_size
- file_modification_time
- etc.

In [0]:
from pyspark.sql.functions import col, current_timestamp, input_file_name

df_orders = df_orders.withColumn("ingesttime", current_timestamp()) \
                      .withColumn("source_file", col("_metadata.file_name"))
                    #   .withColumn("source_file", col("_metadata").getField("file_name"))

df_orders.writeStream.format("parquet")\
                      .outputMode("append")\
                      .option("checkpointLocation", f"abfss://bronze-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/checkPoint_{p_file_name.capitalize()}")\
                      .option("path", f"abfss://bronze-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/{p_file_name}")\
                      .trigger(once=True)\
                      .toTable(f"medallion_arc_e2e.bronze.{p_file_name}")
                      # .start()

In [0]:
# df = spark.read.format("parquet")\
#                 .load(f"abfss://bronze-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/{p_file_name}")

# display(df)

In [0]:
# df = spark.read.format("parquet").load("abfss://bronze-e2e-eus-p01@adlsdemodatabrickseus202.blob.core.windows.net/orders")

In [0]:
%sql
SELECT * FROM medallion_arc_e2e.bronze.orders